# quant_cn — research entry

Working notebook (plan 01 phase 6). Imports only from `quant_cn`; reads the lake only through `LakeQuery` / `PitAligner`. The lake root comes from `.env` (`QUANT_CN_LAKE_ROOT`). Themed notebooks live in `notebooks/` as `YYYYMMDD_<topic>.ipynb`.

In [ ]:
from quant_cn.core.config import Config
from quant_cn.core.date_codec import DateCodec
from quant_cn.lake.lake_catalog import LakeCatalog
from quant_cn.lake.lake_query import LakeQuery
from quant_cn.lake.pit_aligner import PitAligner
from quant_cn.visualization.chart_theme import ChartTheme
from quant_cn.visualization.price_chart import PriceChart

config = Config.load()
catalog = LakeCatalog(config.lake.root, config.datasets)
query = LakeQuery(catalog)
codec = DateCodec()
config.lake.root

## Adjusted prices, August 2026 sample

`derived.prices_adj` = `close × adj_factor` (backward adjusted); suspended days are absent.

In [ ]:
START, END = "20260801", "20260831"
TICKERS = ["000001.SZ", "600519.SH"]

prices = query.read_prices(START, END, TICKERS)
prices.groupby("ts_code").agg(days=("trade_date", "count"), first=("close_adj", "first"), last=("close_adj", "last"))

In [ ]:
theme = ChartTheme.from_mode("light")
PriceChart(theme, "600519.SH", codec).render(prices).show()

In [ ]:
PriceChart(theme, "000001.SZ", codec).render(prices).show()

## Point-in-time fundamentals

A value is visible from the first trading day after its announcement (`PitAligner`, D-006/D-036).

In [ ]:
pit = PitAligner(catalog, query, codec).read_aligned(START, END, TICKERS, ["roe", "total_profit", "n_cashflow_act"])
pit.groupby(["ts_code", "field"]).agg(first_visible=("trade_date", "min"), end_date=("end_date", "last"), ann_date=("ann_date", "last"), value=("value", "last"))

## Recent runs (`meta.run_log`)

In [ ]:
query.sql("SELECT run_id, kind, status, started_at, finished_at, message FROM meta.run_log WHERE dataset IS NULL ORDER BY started_at DESC LIMIT 5")

In [ ]:
catalog.close()